# Phase 5 · Submission S3 — Architecture D (Dilated Residual TCN) + Bottleneck Adapters, LP-FT, multi-label

**Task:** Legal Statute Identification — predict the set of relevant statute IDs (100 IDs, 0–99) for each case. **Primary metric:** Macro-F1 over statutes (secondary Micro-F1, subset accuracy).

**Method** (same multi-label pipeline as S1/S2, different Phase-1 model and adaptation technique)
1. **Backbone:** Phase-1 Architecture D (10 dilated residual TCN blocks, 1024 channels, kernel 3, dilations 1…512, 114.2M parameters), all Phase-1 weights frozen; dropout off. The causal convolution is computed exactly as in Phase 1, as one matrix multiply per kernel tap (fast on GPU).
2. **Adaptation — bottleneck adapters (Houlsby-style):** after every TCN block, `x + W_up·GELU(W_down·x)` with a 64-dimensional bottleneck; `W_up = 0` at start, so the adapted model initially equals Phase 1 (checked in Cell 7). Only the adapters (≈1.3M parameters) and the head are trained.
3. **Input / long documents:** case-sensitive text; `<SECTION>` → "Section", `<ACT>` → "Act", `<ENTITY>` removed; chunks of 510 BPE tokens, **up to 8 per case**, **evidence-based selection** for longer cases (first + last + chunks with the most "Section"/"Act" mentions).
4. **Multi-label head:** each chunk → [mean of token states ; final state]; case vector = **mean and max over chunks**, standardised → linear layer with **100 sigmoid outputs**.
5. **Loss:** binary cross-entropy with per-statute positive weight √(negatives/positives), capped at 10.
6. **LP-FT:** linear probe on frozen Phase-1 case vectors → copied into the model ("epoch 0") → adapters (learning rate 3e-4) + head (1e-4) fine-tuned for 3 epochs; best of epochs 0–3 on dev kept. (In Phase 4 the same adapters with a random head stayed at a constant prediction for most of training; the probe start avoids that.)
7. **Thresholds:** per statute, tuned on dev (cross-fitted dev estimate; global threshold reported for comparison); every case gets at least one statute.
8. **No leakage:** train only for training; dev for selection and thresholds; test labels never read.
9. **Analysis:** per-statute F1 vs frequency, F1 by number of statutes, confused pairs, section comparison; logits of every case saved to `outputs/teacher_logits.pt`.

**Before running (Kaggle)**
- Accelerator: **GPU T4 ×2** · Add-ons → Secrets: **`HF`** · Internet on.
- Add Input → datasets: **`ruchikhemka/data-archd`** (Architecture D) and **`rohitbatra2004/cs781-phase1-tokenized-corpus`** (tokenizer).
- Expected time ≈ 3–4 hours. Checkpoints every 1,000 steps and every epoch; tokens and features cached; to resume, add this notebook's previous output as an input and run again.

Outputs: `/kaggle/working/P5_S3_D_TCN_Adapters/` — upload `submission.zip`.

In [ ]:
# Cell 1 — Imports, random seed and device.
# Every later cell uses the names defined here.
import os, json, glob, math, time, random, shutil, zipfile
from pathlib import Path
from collections import Counter

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import hashlib, re
from sklearn.metrics import f1_score, precision_score, recall_score, classification_report, confusion_matrix
from tokenizers import Tokenizer
from huggingface_hub import snapshot_download

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("PyTorch", torch.__version__, "| device:", DEVICE, "| GPUs visible:", torch.cuda.device_count())

In [ ]:
# Cell 2 — Paths and hyperparameters for this submission.
# Every setting lives here; no other cell hard-codes a value.
RUN_NAME = "P5_S3_D_TCN_Adapters"
WORK_DIR = Path("/kaggle/working") / RUN_NAME
DATA_DIR = Path("/kaggle/working/data")
CKPT_DIR = WORK_DIR / "checkpoints"
OUT_DIR = WORK_DIR / "outputs"
for folder in (WORK_DIR, DATA_DIR, CKPT_DIR, OUT_DIR):
    folder.mkdir(parents=True, exist_ok=True)
HF_REPO_ID = "Exploration-Lab/CS-781-Capstone"

# Long documents: chunks of 510 BPE tokens (+ [BOS], [EOS] = 512); a setting = (selection rule, max chunks).
CHUNK_TOKENS = 510
MAX_DOC_TOKENS = 64 * CHUNK_TOKENS          # tokens stored per case (first 64 chunks; > 99.9% of cases complete)
MAX_DOC_CHARS = 8 * MAX_DOC_TOKENS          # characters tokenised per case (bounds the time on a few huge cases)
TRAIN_SETTING = ("evidence", 8)             # used for the probe, training and (unless the section analysis wins) test
SECTION_SETTINGS = (("first", 1), ("first", 8), ("even", 8), ("evidence", 8), ("evidence", 16))

# Bottleneck adapters after every TCN residual block.
ADAPTER_DIM = 64

# Multi-label head and loss.
FEATURE_CLAMP = 10.0                        # standardised case features clipped to ±10
DROPOUT = 0.1
POS_WEIGHT_POWER = 0.5                      # positive weight = (negatives / positives) ** 0.5 …
POS_WEIGHT_CAP = 10.0                       # … capped at 10

# Step 1 — linear probe on frozen Phase-1 features.
PROBE_WEIGHT_DECAYS = (1e-5, 1e-4, 1e-3)    # tried; the best on dev (cross-fitted Macro-F1) is used
PROBE_EPOCHS = 60                           # evaluated every 5 epochs; the best is kept
PROBE_LR = 1e-3
PROBE_BATCH = 512

# Step 2 — LP-FT fine-tuning.
EPOCHS = 3
CHUNK_BUDGET = 16                           # chunks (sequences of ≤ 512 tokens) per forward pass in training
EVAL_CHUNK_BUDGET = 32                      # chunks per forward pass in evaluation
GRAD_ACCUM = 2
LR_ADAPTER = 3e-4                           # adapters start at the identity, so they need a larger step
LR_HEAD = 1e-4
WEIGHT_DECAY = 0.01
WARMUP_FRACTION = 0.06
GRAD_CLIP = 1.0
SAVE_EVERY_STEPS = 1000
CHECK_EVERY_STEPS = 2000                    # quick dev check on CHECK_CASES dev cases
CHECK_CASES = 1500
THRESHOLD_GRID = np.round(np.arange(0.05, 0.951, 0.025), 3)
USE_AMP = DEVICE.type == "cuda"
print("Run:", RUN_NAME, "| outputs:", WORK_DIR)

In [ ]:
# Cell 3 — Download Phase_5_LSI from Hugging Face using the Kaggle secret "HF".
from kaggle_secrets import UserSecretsClient
snapshot_download(repo_id=HF_REPO_ID, repo_type="dataset", token=UserSecretsClient().get_secret("HF"),
                  local_dir=str(DATA_DIR), allow_patterns=["Phase_5_LSI/**"])
P5_DIR = DATA_DIR / "Phase_5_LSI"
for name in ("Train/train.json", "Train/dev.json", "Test/test.json"):
    assert (P5_DIR / name).is_file(), f"Missing {name}"
print("Phase 5 data ready in", P5_DIR)

In [ ]:
# Cell 4 — Load the splits, replace the masked placeholders, remove duplicate training texts, build 0/1 label matrices.
# Text stays case-sensitive (cased tokenizer). Test labels do not exist in the file and are never needed.
PLACEHOLDERS = ((re.compile(r"<\s*section\s*>", re.I), " Section "), (re.compile(r"<\s*act\s*>", re.I), " Act "),
                (re.compile(r"<\s*entity\s*>", re.I), " "))

def normalise(text):
    for pattern, replacement in PLACEHOLDERS:
        text = pattern.sub(replacement, text)
    return " ".join(text.split())

raw_train = json.load(open(P5_DIR / "Train/train.json"))
raw_dev = json.load(open(P5_DIR / "Train/dev.json"))
raw_test = json.load(open(P5_DIR / "Test/test.json"))
LABEL_IDS = sorted({int(l) for d in raw_train + raw_dev for l in d["labels"]})      # valid statute IDs
LABEL_INDEX = {l: j for j, l in enumerate(LABEL_IDS)}
N_LABELS = len(LABEL_IDS)

def label_vector(labels):
    v = np.zeros(N_LABELS, dtype=np.float32)
    v[[LABEL_INDEX[int(l)] for l in labels]] = 1.0
    return v

train_cases, seen = [], set()
for d in raw_train:
    text = normalise(d["text"])
    key = hashlib.md5(text.encode()).hexdigest()
    if key in seen:                       # identical training texts: keep the first occurrence
        continue
    seen.add(key)
    train_cases.append({"id": str(d["id"]), "text": text, "labels": label_vector(d["labels"])})
dev_cases = [{"id": str(d["id"]), "text": normalise(d["text"]), "labels": label_vector(d["labels"])} for d in raw_dev]
test_cases = [{"id": str(d["id"]), "text": normalise(d["text"])} for d in raw_test]
del raw_train, raw_dev, raw_test
Y_train = np.stack([c["labels"] for c in train_cases])
Y_dev = np.stack([c["labels"] for c in dev_cases])
print(f"{N_LABELS} statute IDs ({LABEL_IDS[0]}–{LABEL_IDS[-1]}) | train {len(train_cases):,} unique cases | "
      f"dev {len(dev_cases):,} | test {len(test_cases):,}")
print(f"labels per case: train {Y_train.sum(1).mean():.2f}, dev {Y_dev.sum(1).mean():.2f} | "
      f"rarest statute has {int(Y_train.sum(0).min())} train / {int(Y_dev.sum(0).min())} dev cases")
print("Example after normalisation:", train_cases[0]["text"][:300])

In [ ]:
# Cell 5 — Tokenise every case (cased Phase-1 BPE, first 64 chunks kept, cached) and define the chunk-selection rules:
# "first" = first k chunks, "even" = k evenly spaced chunks, "evidence" = first + last + chunks with most "Section"/"Act".
tokenizer = Tokenizer.from_file(sorted(glob.glob("/kaggle/input/**/tokenizer/tokenizer.json", recursive=True))[0])
PAD_ID, BOS_ID, EOS_ID = (tokenizer.token_to_id(t) for t in ("[PAD]", "[BOS]", "[EOS]"))
assert (PAD_ID, BOS_ID, EOS_ID) == (0, 2, 3)
EVIDENCE_IDS = np.array(sorted({tokenizer.token_to_id(t) for t in ("ĠSection", "Section", "ĠAct", "Act")} - {None}))
SPLITS = (("train", train_cases), ("dev", dev_cases), ("test", test_cases))

def encode_cases(cases):
    out = []
    for start in range(0, len(cases), 256):
        enc = tokenizer.encode_batch([c["text"][:MAX_DOC_CHARS] for c in cases[start:start + 256]])
        out += [np.array(e.ids[:MAX_DOC_TOKENS], dtype=np.uint16) for e in enc]
    return out

cache = WORK_DIR / "tokens_cache.pt"
if not cache.exists():
    previous = sorted(glob.glob(f"/kaggle/input/**/{RUN_NAME}/tokens_cache.pt", recursive=True))
    if previous:
        shutil.copy(previous[0], cache)
if cache.exists():
    tokens = torch.load(cache, weights_only=False)
    print("Loaded cached tokens from", cache)
else:
    start_time = time.time()
    tokens = {name: encode_cases(cases) for name, cases in SPLITS}
    torch.save(tokens, cache)
    print(f"Tokenised all cases in {(time.time() - start_time) / 60:.1f} min → {cache}")
for name, cases in SPLITS:
    assert len(tokens[name]) == len(cases)

def select_chunks(ids, setting):
    """Token lists ([BOS] … [EOS]) of the chunks a setting uses for one case, in document order."""
    kind, k = setting
    pieces = [ids[s:s + CHUNK_TOKENS] for s in range(0, len(ids), CHUNK_TOKENS)] or [ids[:0]]
    if len(pieces) <= k:
        chosen = range(len(pieces))
    elif kind == "first":
        chosen = range(k)
    elif kind == "even":
        chosen = sorted(set(np.linspace(0, len(pieces) - 1, k).round().astype(int).tolist()))
    else:                                                    # "evidence" (k >= 3)
        score = [int(np.isin(p, EVIDENCE_IDS).sum()) for p in pieces]
        middle = sorted(range(1, len(pieces) - 1), key=lambda i: (-score[i], i))[:k - 2]
        chosen = sorted([0, len(pieces) - 1] + middle)
    return [[BOS_ID] + pieces[i].tolist() + [EOS_ID] for i in chosen]

def n_chunks(split, i, setting):
    return min(max(1, -(-len(tokens[split][i]) // CHUNK_TOKENS)), setting[1])

for name, _ in SPLITS:
    full = np.array([max(1, -(-len(t) // CHUNK_TOKENS)) for t in tokens[name]])
    print(f"{name}: chunks per case median {int(np.median(full))}, p90 {int(np.percentile(full, 90))} | "
          f"fully covered by {TRAIN_SETTING[1]} chunks: {100 * np.mean(full <= TRAIN_SETTING[1]):.1f}%")

In [ ]:
# Cell 6 — Rebuild the Phase-1 Architecture D TCN and strictly load its pretrained weights.
# Weights and configuration both come from checkpoints/best.pt (keys "model_state_dict" and "model_config").
d_paths = sorted(glob.glob("/kaggle/input/**/architecture_D/checkpoints/best.pt", recursive=True))
assert d_paths, "Add the dataset ruchikhemka/data-archd as an input."
D_CKPT = torch.load(d_paths[0], map_location="cpu", mmap=True, weights_only=False)
D_CONFIG = D_CKPT["model_config"]
assert D_CONFIG["embedding_dim"] == D_CONFIG["channels"] and D_CONFIG["convs_per_block"] == 2

class WeightNormConvParams(nn.Module):
    # Holds the Phase-1 weight-normalised convolution exactly as saved: weight = g * v / ||v||.
    def __init__(self, channels, kernel_size):
        super().__init__()
        self.weight_g = nn.Parameter(torch.ones(channels, 1, 1))
        self.weight_v = nn.Parameter(0.02 * torch.randn(channels, channels, kernel_size))  # overwritten by Phase-1 weights
        self.bias = nn.Parameter(torch.zeros(channels))

    def weight(self):
        return self.weight_g * self.weight_v / self.weight_v.norm(dim=(1, 2), keepdim=True)

class CausalConv1d(nn.Module):
    # Dilated causal convolution computed as one matrix multiply per kernel tap, on [batch, time, channels].
    # Mathematically identical to Phase 1's left-padded Conv1d, but a tap whose shift is >= the sequence length
    # only ever reads padding zeros and is skipped. Sentences are <= 148 positions while dilations reach 512,
    # so this avoids the very slow cuDNN path for large dilations on short inputs.
    def __init__(self, channels, kernel_size, dilation):
        super().__init__()
        self.kernel_size, self.dilation = kernel_size, dilation
        self.conv = WeightNormConvParams(channels, kernel_size)

    def forward(self, x):
        T = x.size(1)
        w = self.conv.weight()
        out = self.conv.bias
        for j in range(self.kernel_size):
            shift = (self.kernel_size - 1 - j) * self.dilation
            if shift >= T:
                continue
            shifted = F.pad(x, (0, 0, shift, 0))[:, :T] if shift > 0 else x
            out = out + shifted @ w[:, :, j].t()
        return out

class TCNResidualBlock(nn.Module):
    def __init__(self, channels, kernel_size, dilation, dropout):
        super().__init__()
        self.conv1 = CausalConv1d(channels, kernel_size, dilation)
        self.conv2 = CausalConv1d(channels, kernel_size, dilation)
        self.dropout1 = nn.Dropout(dropout)
        self.dropout2 = nn.Dropout(dropout)

    def forward(self, x):
        out = self.dropout1(F.relu(self.conv1(x)))
        out = self.dropout2(F.relu(self.conv2(out)))
        return F.relu(out + x)

class ArchitectureDBackbone(nn.Module):
    # Phase-1 TCN language model without the vocabulary projection.
    # forward(input_ids) returns the state of every token: [batch, tokens, channels].
    def __init__(self, cfg):
        super().__init__()
        self.token_embedding = nn.Embedding(cfg["vocab_size"], cfg["embedding_dim"])
        self.tcn_blocks = nn.ModuleList([TCNResidualBlock(cfg["channels"], cfg["kernel_size"], d, cfg["dropout"])
                                         for d in cfg["dilations"]])

    def forward(self, input_ids):
        x = self.token_embedding(input_ids)
        for block in self.tcn_blocks:
            x = block(x)
        return x

backbone = ArchitectureDBackbone(D_CONFIG)
backbone.load_state_dict(D_CKPT["model_state_dict"], strict=True)
del D_CKPT

n_backbone = sum(p.numel() for p in backbone.parameters())
assert n_backbone == int(D_CONFIG["estimated_parameter_count"]), "Parameter count differs from Phase-1 config"
receptive_field = 1 + 2 * (D_CONFIG["kernel_size"] - 1) * sum(D_CONFIG["dilations"])
print(json.dumps({k: v for k, v in D_CONFIG.items() if not isinstance(v, (dict, list)) or k == "dilations"}, indent=2))
print(f"Loaded Phase-1 Architecture D from {Path(d_paths[0]).parent} — {n_backbone:,} parameters, "
      f"receptive field {receptive_field} tokens")

In [ ]:
# Cell 7 — Bottleneck adapters: x + W_up·GELU(W_down·x) after every frozen TCN block (W_up = 0 at start).
class BottleneckAdapter(nn.Module):
    def __init__(self, channels, bottleneck):
        super().__init__()
        self.adapter_down = nn.Linear(channels, bottleneck)
        self.adapter_up = nn.Linear(bottleneck, channels)
        nn.init.zeros_(self.adapter_up.weight)
        nn.init.zeros_(self.adapter_up.bias)

    def forward(self, x):
        return x + self.adapter_up(F.gelu(self.adapter_down(x)))

class BlockWithAdapter(nn.Module):
    def __init__(self, block, channels, bottleneck):
        super().__init__()
        self.block = block
        self.adapter = BottleneckAdapter(channels, bottleneck)

    def forward(self, x):
        return self.adapter(self.block(x))

for p in backbone.parameters():
    p.requires_grad_(False)
probe_ids = torch.randint(4, 1000, (2, 16))
with torch.no_grad():
    before = backbone.eval()(probe_ids)
backbone.tcn_blocks = nn.ModuleList([BlockWithAdapter(b, D_CONFIG["channels"], ADAPTER_DIM) for b in backbone.tcn_blocks])
with torch.no_grad():
    assert torch.allclose(before, backbone(probe_ids), atol=1e-5), "Adapters must start as the identity"
adapted_backbone = backbone.to(DEVICE).train()
HIDDEN = D_CONFIG["channels"]

# Check on the device that every adapter receives a gradient.
with torch.no_grad():
    for m in adapted_backbone.modules():
        if isinstance(m, BottleneckAdapter):
            m.adapter_up.weight.normal_(0, 1e-3)                 # temporary non-zero so W_down also gets a gradient
adapted_backbone(probe_ids.to(DEVICE)).float().pow(2).mean().backward()
assert all(p.grad is not None and p.grad.abs().sum() > 0 for n, p in adapted_backbone.named_parameters() if p.requires_grad)
with torch.no_grad():
    for m in adapted_backbone.modules():
        if isinstance(m, BottleneckAdapter):
            m.adapter_up.weight.zero_()                           # back to the identity before training
adapted_backbone.zero_grad(set_to_none=True)
n_tuned = sum(p.numel() for p in adapted_backbone.parameters() if p.requires_grad)
print(f"Adapters after {len(adapted_backbone.tcn_blocks)} TCN blocks — {n_tuned:,} trainable backbone parameters "
      f"({100 * n_tuned / n_backbone:.2f}% of the backbone)")

In [ ]:
# Cell 8 — Multi-label case classifier: per chunk [mean ; final state] → case = [mean ; max] over chunks
# → standardise (statistics from the probe, Cell 11) → linear layer with one sigmoid output per statute.
class CaseClassifier(nn.Module):
    def __init__(self, backbone, hidden, n_labels):
        super().__init__()
        self.backbone = backbone
        self.register_buffer("feat_mean", torch.zeros(4 * hidden))
        self.register_buffer("feat_std", torch.ones(4 * hidden))
        self.dropout = nn.Dropout(DROPOUT)
        self.classifier = nn.Linear(4 * hidden, n_labels)

    def case_features(self, ids, chunk_mask, standardise=True):
        """ids [B, K, T], chunk_mask [B, K] → case vectors [B, 4·hidden] (standardised and clipped, or raw)."""
        B, K, T = ids.shape
        flat_ids, valid = ids.view(B * K, T), chunk_mask.view(-1)
        token_mask = flat_ids[valid].ne(PAD_ID)
        states = self.backbone(flat_ids[valid]).float()
        m = token_mask.unsqueeze(-1).float()
        mean = (states * m).sum(1) / m.sum(1).clamp(min=1.0)
        last = states[torch.arange(states.size(0), device=states.device), token_mask.sum(1) - 1]
        vecs = torch.zeros(B * K, 2 * states.size(-1), device=ids.device)
        vecs[valid] = torch.cat([mean, last], dim=-1)
        vecs = vecs.view(B, K, -1)
        w = chunk_mask.float().unsqueeze(-1)
        case_mean = (vecs * w).sum(1) / w.sum(1).clamp(min=1.0)
        case_max = vecs.masked_fill(~chunk_mask.unsqueeze(-1), -1e4).max(1).values
        case = torch.cat([case_mean, case_max], dim=-1)
        if not standardise:
            return case
        return ((case - self.feat_mean) / self.feat_std).clamp(-FEATURE_CLAMP, FEATURE_CLAMP)

    def forward(self, ids, chunk_mask):
        return self.classifier(self.dropout(self.case_features(ids, chunk_mask)))      # [B, n_labels] logits

class FeatureRunner(nn.Module):
    """Wraps the model so that DataParallel can compute raw case vectors on both GPUs."""
    def __init__(self, model):
        super().__init__()
        self.model = model

    def forward(self, ids, chunk_mask):
        return self.model.case_features(ids, chunk_mask, standardise=False)

torch.manual_seed(SEED)
model = CaseClassifier(adapted_backbone, HIDDEN, N_LABELS).to(DEVICE)
TRAINABLE_NAMES = [n for n, p in model.named_parameters() if p.requires_grad]
SAVED_NAMES = TRAINABLE_NAMES + ["feat_mean", "feat_std"]
n_trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
n_total = sum(p.numel() for p in model.parameters())
N_GPUS = torch.cuda.device_count()
train_model = nn.DataParallel(model) if N_GPUS > 1 else model
print(f"Trainable parameters: {n_trainable:,} of {n_total:,} ({100 * n_trainable / n_total:.2f}%) | GPUs used: {max(N_GPUS, 1)}")

In [ ]:
# Cell 9 — Batches by a chunk budget (short cases share a batch, long ones get their own) and per-statute positive weights.
def make_batch(indices, split, setting):
    per_case = [select_chunks(tokens[split][i], setting) for i in indices]
    K = max(len(c) for c in per_case)
    T = max(len(x) for c in per_case for x in c)
    ids = torch.full((len(indices), K, T), PAD_ID, dtype=torch.long)
    mask = torch.zeros(len(indices), K, dtype=torch.bool)
    for b, chunks in enumerate(per_case):
        for k, x in enumerate(chunks):
            ids[b, k, :len(x)] = torch.tensor(x)
            mask[b, k] = True
    return ids.to(DEVICE), mask.to(DEVICE)

def budget_batches(indices, split, setting, budget):
    batches, current, used = [], [], 0
    for i in indices:
        n = n_chunks(split, i, setting)
        if current and used + n > budget:
            batches.append(current); current, used = [], 0
        current.append(i); used += n
    return batches + ([current] if current else [])

def epoch_batches(epoch):
    order = list(range(len(train_cases)))
    random.Random(SEED + epoch).shuffle(order)                 # same batches after a resume
    return budget_batches(order, "train", TRAIN_SETTING, CHUNK_BUDGET)

positives = Y_train.sum(0)
POS_WEIGHT = np.clip(((len(Y_train) - positives) / positives) ** POS_WEIGHT_POWER, 1.0, POS_WEIGHT_CAP).astype(np.float32)
POS_WEIGHT_T = torch.tensor(POS_WEIGHT, device=DEVICE)
ids, mask = make_batch([0, 1], "train", TRAIN_SETTING)
print("Example batch:", tuple(ids.shape), f"| {len(epoch_batches(0)):,} training batches per epoch")
print(f"Positive weights: min {POS_WEIGHT.min():.2f}, median {np.median(POS_WEIGHT):.2f}, max {POS_WEIGHT.max():.2f}")

In [ ]:
# Cell 10 — Multi-label metrics and threshold strategies (global, per-statute, cross-fitted per-statute) on dev.
def per_label_f1(Y, pred):
    tp = (Y * pred).sum(0); fp = ((1 - Y) * pred).sum(0); fn = (Y * (1 - pred)).sum(0)
    return 2 * tp / np.maximum(2 * tp + fp + fn, 1)

def predict_sets(P, thresholds):
    """0/1 predictions; a case with nothing above its thresholds gets its relatively most likely statute."""
    pred = (P >= thresholds).astype(np.float32)
    empty = pred.sum(1) == 0
    pred[np.where(empty)[0], np.argmax(P[empty] / thresholds, axis=1)] = 1.0
    return pred

def all_scores(Y, pred):
    tp, fp, fn = (Y * pred).sum(), ((1 - Y) * pred).sum(), (Y * (1 - pred)).sum()
    tp_l = (Y * pred).sum(0)
    return {"macro_f1": float(per_label_f1(Y, pred).mean()), "micro_f1": float(2 * tp / max(2 * tp + fp + fn, 1)),
            "subset_accuracy": float((pred == Y).all(1).mean()),
            "macro_precision": float((tp_l / np.maximum(pred.sum(0), 1)).mean()),
            "macro_recall": float((tp_l / np.maximum(Y.sum(0), 1)).mean()),
            "labels_per_case": float(pred.sum(1).mean())}

def tune_thresholds(P, Y):
    """Per statute: the grid threshold with the highest F1 on (P, Y)."""
    best_t, best_f = np.full(P.shape[1], 0.5), np.full(P.shape[1], -1.0)
    for t in THRESHOLD_GRID:
        f = per_label_f1(Y, (P >= t).astype(np.float32))
        better = f > best_f
        best_t[better], best_f[better] = t, f[better]
    return best_t

def tune_global(P, Y):
    return max(THRESHOLD_GRID, key=lambda t: all_scores(Y, predict_sets(P, np.full(P.shape[1], t)))["macro_f1"])

def crossfit_predictions(P, Y):
    """Honest per-statute thresholds: tuned on one half of dev, applied to the other half, and vice versa."""
    halves = np.array_split(np.random.RandomState(SEED).permutation(len(P)), 2)
    pred = np.zeros_like(P)
    for a, b in ((0, 1), (1, 0)):
        pred[halves[b]] = predict_sets(P[halves[b]], tune_thresholds(P[halves[a]], Y[halves[a]]))
    return pred

@torch.no_grad()
def predict_logits(net, split, setting=TRAIN_SETTING, limit=None):
    net.eval()
    n = len(tokens[split]) if limit is None else min(limit, len(tokens[split]))
    out = []
    for batch in budget_batches(range(n), split, setting, EVAL_CHUNK_BUDGET):
        with torch.autocast(device_type=DEVICE.type, dtype=torch.float16, enabled=USE_AMP):
            out.append(net(*make_batch(batch, split, setting)).float().cpu())
    return torch.cat(out)

def evaluate_probs(P, Y):
    g = tune_global(P, Y)
    return {"global": {**all_scores(Y, predict_sets(P, np.full(P.shape[1], g))), "threshold": float(g)},
            "per_label_crossfit": all_scores(Y, crossfit_predictions(P, Y)),
            "per_label_full_dev": all_scores(Y, predict_sets(P, tune_thresholds(P, Y)))}

def evaluate(net, setting=TRAIN_SETTING, limit=None):
    P = torch.sigmoid(predict_logits(net, "dev", setting, limit)).numpy()
    return evaluate_probs(P, Y_dev[:len(P)]), P

def describe(result):
    c, g = result["per_label_crossfit"], result["global"]
    return (f"Macro-F1 {c['macro_f1']:.4f} (per-statute thresholds, cross-fitted) | global threshold {g['threshold']:.3f}: "
            f"{g['macro_f1']:.4f} | Micro-F1 {c['micro_f1']:.4f} | subset acc {c['subset_accuracy']:.4f} | "
            f"labels/case {c['labels_per_case']:.2f}")
print("Metrics ready (dev only).")

In [ ]:
# Cell 11 — Step 1, linear probe (the "LP" of LP-FT): case vectors of every train and dev case from the frozen Phase-1
# model (Phase-1 weights) → 100-output linear layer trained with the weighted BCE → copied into the model as "epoch 0".
def trainable_state():
    full = model.state_dict()
    return {n: full[n].detach().cpu() for n in SAVED_NAMES}

LATEST, BEST = CKPT_DIR / "latest.pt", CKPT_DIR / "best.pt"

@torch.no_grad()
def raw_case_features(split):
    runner = FeatureRunner(model)
    runner = nn.DataParallel(runner) if N_GPUS > 1 else runner
    runner.eval()
    out = []
    for batch in budget_batches(range(len(tokens[split])), split, TRAIN_SETTING, EVAL_CHUNK_BUDGET):
        with torch.autocast(device_type=DEVICE.type, dtype=torch.float16, enabled=USE_AMP):
            out.append(runner(*make_batch(batch, split, TRAIN_SETTING)).float().cpu())
    return torch.cat(out)

feature_cache = WORK_DIR / "frozen_features.pt"
if not feature_cache.exists():
    previous = sorted(glob.glob(f"/kaggle/input/**/{RUN_NAME}/frozen_features.pt", recursive=True))
    if previous:
        shutil.copy(previous[0], feature_cache)
if feature_cache.exists():
    frozen = torch.load(feature_cache)
    print("Loaded cached frozen features from", feature_cache)
else:
    start_time = time.time()
    frozen = {"train": raw_case_features("train").half(), "dev": raw_case_features("dev").half()}
    torch.save(frozen, feature_cache)
    print(f"Frozen Phase-1 case vectors of {len(frozen['train']):,} train + {len(frozen['dev']):,} dev cases "
          f"in {(time.time() - start_time) / 60:.1f} min")
assert frozen["train"].shape[0] == len(train_cases) and frozen["dev"].shape[0] == len(dev_cases)

feat_mean = frozen["train"].float().mean(0)
feat_std = frozen["train"].float().std(0).clamp(min=1e-4)
def standardise(x):
    return ((x.float() - feat_mean) / feat_std).clamp(-FEATURE_CLAMP, FEATURE_CLAMP).to(DEVICE)
X_tr, X_dv = standardise(frozen["train"]), standardise(frozen["dev"])
Y_tr = torch.tensor(Y_train, device=DEVICE)
prior = Y_train.mean(0).clip(1e-4, 1 - 1e-4)

def train_probe(weight_decay):
    torch.manual_seed(SEED)
    layer = nn.Linear(X_tr.size(1), N_LABELS).to(DEVICE)
    with torch.no_grad():
        layer.weight.zero_(); layer.bias.copy_(torch.tensor(np.log(prior / (1 - prior)), dtype=torch.float32))
    optimizer = torch.optim.AdamW(layer.parameters(), lr=PROBE_LR, weight_decay=weight_decay)
    best = (-1.0, None, 0)
    for epoch in range(1, PROBE_EPOCHS + 1):
        for idx in torch.randperm(len(X_tr), device=DEVICE).split(PROBE_BATCH):
            loss = F.binary_cross_entropy_with_logits(layer(X_tr[idx]), Y_tr[idx], pos_weight=POS_WEIGHT_T)
            optimizer.zero_grad(); loss.backward(); optimizer.step()
        if epoch % 5 == 0:
            with torch.no_grad():
                P = torch.sigmoid(layer(X_dv)).cpu().numpy()
            score = all_scores(Y_dev, crossfit_predictions(P, Y_dev))["macro_f1"]
            if score > best[0]:
                best = (score, {k: v.detach().clone() for k, v in layer.state_dict().items()}, epoch)
    return best

probe_results, best_probe = {}, None
for wd in PROBE_WEIGHT_DECAYS:
    start_time = time.time()
    score, state_dict, epoch = train_probe(wd)
    probe_results[str(wd)] = {"dev_macro_f1_crossfit": round(score, 4), "best_epoch": epoch}
    print(f"  probe weight decay {wd}: dev Macro-F1 {score:.4f} (cross-fitted) at probe epoch {epoch} | {time.time() - start_time:.0f} s")
    if best_probe is None or score > best_probe[0]:
        best_probe = (score, wd, state_dict)
PROBE_F1, PROBE_WD, probe_state = best_probe

with torch.no_grad():                                      # copy the probe into the model
    model.feat_mean.copy_(feat_mean); model.feat_std.copy_(feat_std)
    model.classifier.weight.copy_(probe_state["weight"]); model.classifier.bias.copy_(probe_state["bias"])
probe_dev, probe_P = evaluate(train_model)
print(f"Probe (weight decay {PROBE_WD}) copied into the model → epoch 0: {describe(probe_dev)}")
assert abs(probe_dev["per_label_crossfit"]["macro_f1"] - PROBE_F1) < 0.02, "The model does not reproduce the probe"
torch.save({"trainable": trainable_state(), "epoch": 0, "dev": probe_dev}, BEST)      # epoch 0 = linear probe only
del X_tr, X_dv, Y_tr
if DEVICE.type == "cuda":
    torch.cuda.empty_cache()

In [ ]:
# Cell 12 — Step 2 (the "FT" of LP-FT): optimiser, learning-rate schedule and checkpoint resume.
adapter_params = [p for n, p in model.named_parameters() if p.requires_grad and n.startswith("backbone.")]   # adapters
head_params = [p for n, p in model.named_parameters() if p.requires_grad and not n.startswith("backbone.")]   # classifier
optimizer = torch.optim.AdamW([{"params": adapter_params, "lr": LR_ADAPTER}, {"params": head_params, "lr": LR_HEAD}],
                              weight_decay=WEIGHT_DECAY)
steps_per_epoch = len(epoch_batches(0))
total_updates = EPOCHS * math.ceil(steps_per_epoch / GRAD_ACCUM)
warmup_updates = int(WARMUP_FRACTION * total_updates)
scheduler = torch.optim.lr_scheduler.LambdaLR(optimizer, lambda u: (u + 1) / max(warmup_updates, 1) if u < warmup_updates
                                              else max(0.0, (total_updates - u) / max(total_updates - warmup_updates, 1)))
scaler = torch.amp.GradScaler("cuda", enabled=USE_AMP)

if not LATEST.exists():
    # Resume only from a run of this notebook with the same saved parameter names.
    for previous in sorted(glob.glob(f"/kaggle/input/**/{RUN_NAME}/checkpoints/latest.pt", recursive=True)):
        if set(torch.load(previous, map_location="cpu", weights_only=False)["trainable"]) == set(SAVED_NAMES):
            shutil.copy(previous, LATEST)
            if Path(previous).with_name("best.pt").exists():
                shutil.copy(Path(previous).with_name("best.pt"), BEST)
            print("Copied checkpoints from", Path(previous).parent)
            break

state = {"epoch": 0, "step": 0, "best_f1": probe_dev["per_label_crossfit"]["macro_f1"], "best_epoch": 0,
         "history": [], "train_seconds": 0.0}
if LATEST.exists():
    ckpt = torch.load(LATEST, map_location="cpu", weights_only=False)
    model.load_state_dict(ckpt["trainable"], strict=False)
    optimizer.load_state_dict(ckpt["optimizer"]); scheduler.load_state_dict(ckpt["scheduler"]); scaler.load_state_dict(ckpt["scaler"])
    state = ckpt["state"]
    print(f"Resumed at epoch {state['epoch'] + 1}, step {state['step']} (best dev Macro-F1 {state['best_f1']:.4f})")
else:
    print("No checkpoint found — fine-tuning starts from the Phase-1 weights + the probe classifier.")
print(f"≈{steps_per_epoch:,} steps per epoch, {total_updates:,} optimiser updates in total")

In [ ]:
# Cell 13 — Fine-tuning the adapters + classifier with the weighted BCE; the frozen TCN runs without dropout.
# Every epoch ends with a full dev evaluation; best.pt keeps the best of epoch 0 (probe), 1, …, EPOCHS.
def save_latest():
    torch.save({"trainable": trainable_state(), "optimizer": optimizer.state_dict(), "scheduler": scheduler.state_dict(),
                "scaler": scaler.state_dict(), "state": state}, LATEST)

if DEVICE.type == "cuda":
    torch.cuda.reset_peak_memory_stats()
while state["epoch"] < EPOCHS:
    epoch = state["epoch"]
    batches = epoch_batches(epoch)
    train_model.train()
    model.backbone.eval()
    start_time, loss_sum, loss_count = time.time(), 0.0, 0
    for step in range(state["step"], len(batches)):
        ids, mask = make_batch(batches[step], "train", TRAIN_SETTING)
        targets = torch.tensor(Y_train[batches[step]], device=DEVICE)
        with torch.autocast(device_type=DEVICE.type, dtype=torch.float16, enabled=USE_AMP):
            logits = train_model(ids, mask)
        loss = F.binary_cross_entropy_with_logits(logits.float(), targets, pos_weight=POS_WEIGHT_T) / GRAD_ACCUM
        scaler.scale(loss).backward()
        loss_sum, loss_count = loss_sum + loss.item() * GRAD_ACCUM, loss_count + 1
        if (step + 1) % GRAD_ACCUM == 0 or step + 1 == len(batches):
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(adapter_params + head_params, GRAD_CLIP)
            scaler.step(optimizer); scaler.update(); scheduler.step()
            optimizer.zero_grad(set_to_none=True)
        state["step"] = step + 1
        if state["step"] % SAVE_EVERY_STEPS == 0:
            state["train_seconds"] += time.time() - start_time; start_time = time.time()
            save_latest()
            print(f"  epoch {epoch + 1} step {state['step']}/{len(batches)} | mean loss {loss_sum / loss_count:.4f}", flush=True)
        if state["step"] % CHECK_EVERY_STEPS == 0:
            quick, _ = evaluate(train_model, limit=CHECK_CASES)
            print(f"    quick dev check ({CHECK_CASES} cases): {describe(quick)}", flush=True)
            train_model.train(); model.backbone.eval()
    state["train_seconds"] += time.time() - start_time
    dev, _ = evaluate(train_model)
    state["history"].append({"epoch": epoch + 1, "train_loss": loss_sum / max(loss_count, 1), "dev": dev})
    mark = ""
    if dev["per_label_crossfit"]["macro_f1"] > state["best_f1"]:
        state.update(best_f1=dev["per_label_crossfit"]["macro_f1"], best_epoch=epoch + 1)
        torch.save({"trainable": trainable_state(), "epoch": epoch + 1, "dev": dev}, BEST)
        mark = "  <- best"
    state["epoch"], state["step"] = epoch + 1, 0
    save_latest()
    print(f"Epoch {epoch + 1} | loss {loss_sum / max(loss_count, 1):.4f} | dev {describe(dev)}{mark}", flush=True)
print(f"Best dev Macro-F1 {state['best_f1']:.4f} at epoch {state['best_epoch']} (0 = probe only) | "
      f"training time {state['train_seconds'] / 60:.1f} min")

In [ ]:
# Cell 14 — Best epoch on dev: threshold strategies, per-statute F1 vs training frequency, F1 by number of statutes,
# most-missed statutes and most frequent (missed, wrongly predicted) pairs.
best = torch.load(BEST, map_location="cpu", weights_only=False)
model.load_state_dict(best["trainable"], strict=False)
dev_result, P_dev = evaluate(train_model)
print(f"Best epoch {best['epoch']}:")
for name, r in dev_result.items():
    print(f"  {name:19s}: Macro-F1 {r['macro_f1']:.4f} | Macro-P {r['macro_precision']:.4f} | Macro-R {r['macro_recall']:.4f} | "
          f"Micro-F1 {r['micro_f1']:.4f} | subset acc {r['subset_accuracy']:.4f} | labels/case {r['labels_per_case']:.2f}")

thresholds = tune_thresholds(P_dev, Y_dev)
pred_dev = predict_sets(P_dev, thresholds)
f1_label = per_label_f1(Y_dev, pred_dev)
train_freq = Y_train.sum(0)
quartiles = np.quantile(train_freq, [0.25, 0.5, 0.75])
bucket = np.digitize(train_freq, quartiles)
by_frequency = {f"Q{q + 1} (train cases {int(train_freq[bucket == q].min())}–{int(train_freq[bucket == q].max())})":
                round(float(f1_label[bucket == q].mean()), 4) for q in range(4)}
rank_corr = float(np.corrcoef(np.argsort(np.argsort(train_freq)), np.argsort(np.argsort(f1_label)))[0, 1])
print("Per-statute F1 by training-frequency quartile:", by_frequency, f"| rank correlation {rank_corr:.2f}")

n_gold = Y_dev.sum(1)
example_f1 = 2 * (Y_dev * pred_dev).sum(1) / (Y_dev.sum(1) + pred_dev.sum(1))
by_size = {name: round(float(example_f1[sel].mean()), 4) for name, sel in
           (("1 statute", n_gold == 1), ("2-3", (n_gold >= 2) & (n_gold <= 3)), ("4-6", (n_gold >= 4) & (n_gold <= 6)), ("7+", n_gold >= 7))
           if sel.any()}
print("Example-based F1 by number of gold statutes:", by_size)

order = np.argsort(f1_label)
worst = [(LABEL_IDS[j], round(float(f1_label[j]), 3), int(train_freq[j])) for j in order[:10]]
best_labels = [(LABEL_IDS[j], round(float(f1_label[j]), 3), int(train_freq[j])) for j in order[-10:][::-1]]
print("Hardest statutes (id, F1, train cases):", worst)
print("Easiest statutes (id, F1, train cases):", best_labels)
pairs = Counter()
for y, p in zip(Y_dev, pred_dev):
    missed, wrong = np.where((y == 1) & (p == 0))[0], np.where((y == 0) & (p == 1))[0]
    pairs.update((LABEL_IDS[m], LABEL_IDS[w]) for m in missed for w in wrong)
print("Most frequent (missed statute, wrongly predicted statute) pairs:", pairs.most_common(10))
json.dump({"dev": dev_result, "best_epoch": best["epoch"], "history": state["history"], "probe": probe_results,
           "per_label_f1": {str(LABEL_IDS[j]): round(float(f1_label[j]), 4) for j in range(N_LABELS)},
           "thresholds": {str(LABEL_IDS[j]): float(thresholds[j]) for j in range(N_LABELS)},
           "f1_by_train_frequency_quartile": by_frequency, "rank_corr_frequency_f1": rank_corr,
           "example_f1_by_n_statutes": by_size, "confused_pairs": [[a, b, n] for (a, b), n in pairs.most_common(30)]},
          open(OUT_DIR / "dev_metrics.json", "w"), indent=2)

In [ ]:
# Cell 15 — Section analysis on dev: which parts of the document to read (first chunk / first 8 / evenly spaced 8 /
# evidence 8 / evidence 16). The best setting on dev (cross-fitted Macro-F1) is used for the test predictions.
section_results, section_probs = {}, {}
for setting in SECTION_SETTINGS:
    result, P = evaluate(train_model, setting)
    key = f"{setting[0]}_{setting[1]}"
    section_results[key], section_probs[key] = result, P
    print(f"  {key:12s}: {describe(result)}")
best_section = max(section_results, key=lambda k: section_results[k]["per_label_crossfit"]["macro_f1"])
TEST_SETTING = next(s for s in SECTION_SETTINGS if f"{s[0]}_{s[1]}" == best_section)
TEST_THRESHOLDS = tune_thresholds(section_probs[best_section], Y_dev)      # final thresholds: tuned on the whole dev set
print(f"Chosen for test: {best_section} | per-statute thresholds tuned on all dev cases "
      f"(median {np.median(TEST_THRESHOLDS):.3f}, range {TEST_THRESHOLDS.min():.3f}–{TEST_THRESHOLDS.max():.3f})")
json.dump(section_results, open(OUT_DIR / "section_analysis.json", "w"), indent=2)

In [ ]:
# Cell 16 — Predict the test cases, validate the format and build submission.zip.
# prediction.json is a JSON list of {"id": ..., "pred": [statute IDs]} — integers, valid IDs, unique, at least one per case.
test_logits = predict_logits(train_model, "test", TEST_SETTING)
pred_test = predict_sets(torch.sigmoid(test_logits).numpy(), TEST_THRESHOLDS)
prediction = [{"id": c["id"], "pred": [int(LABEL_IDS[j]) for j in np.where(p == 1)[0]]} for c, p in zip(test_cases, pred_test)]
valid = set(LABEL_IDS)
assert len(prediction) == len(test_cases) and len({p["id"] for p in prediction}) == len(prediction)
assert all(p["pred"] and len(set(p["pred"])) == len(p["pred"]) and set(p["pred"]) <= valid and
           all(isinstance(x, int) for x in p["pred"]) for p in prediction)
json.dump(prediction, open(WORK_DIR / "prediction.json", "w"))
with zipfile.ZipFile(WORK_DIR / "submission.zip", "w", zipfile.ZIP_DEFLATED) as zf:
    zf.write(WORK_DIR / "prediction.json", arcname="prediction.json")
with zipfile.ZipFile(WORK_DIR / "submission.zip") as zf:
    assert zf.namelist() == ["prediction.json"]
sizes = np.array([len(p["pred"]) for p in prediction])
print(f"submission.zip: {len(prediction):,} cases | statutes per case mean {sizes.mean():.2f} (dev gold {Y_dev.sum(1).mean():.2f}), "
      f"max {sizes.max()} | distinct statutes predicted {len({x for p in prediction for x in p['pred']})}")

In [ ]:
# Cell 17 — Logits of the final model for every train, dev and test case (for comparisons / ensembling with other runs),
# (same input setting as the test predictions), saved with ids, statute IDs and thresholds.
start_time = time.time()
teacher = {"train": predict_logits(train_model, "train", TEST_SETTING).half(),
           "dev": predict_logits(train_model, "dev", TEST_SETTING).half(), "test": test_logits.half(),
           "ids": {name: [c["id"] for c in cases] for name, cases in SPLITS}, "label_ids": LABEL_IDS,
           "setting": list(TEST_SETTING), "thresholds": TEST_THRESHOLDS.tolist(), "best_epoch": best["epoch"]}
torch.save(teacher, OUT_DIR / "teacher_logits.pt")
train_pred = predict_sets(torch.sigmoid(teacher["train"].float()).numpy(), TEST_THRESHOLDS)
print(f"Logits saved ({(time.time() - start_time) / 60:.1f} min) → {OUT_DIR / 'teacher_logits.pt'} | "
      f"Macro-F1 on train {all_scores(Y_train, train_pred)['macro_f1']:.4f}")

In [ ]:
# Cell 18 — Run summary for the README, the experiment log and results/submissions.csv.
summary = {
    "run": RUN_NAME, "backbone": "Architecture D — dilated residual TCN (Phase 1)",
    "technique": "Bottleneck adapters (64-d) after every TCN block, LP-FT, multi-label BCE, per-statute thresholds",
    "adapter_dim": ADAPTER_DIM, "trainable_params": n_trainable, "total_params": n_total,
    "probe": {"chosen_weight_decay": PROBE_WD, "dev_macro_f1_crossfit": PROBE_F1, "all": probe_results},
    "train_cases": len(train_cases), "dev_cases": len(dev_cases), "epochs": state["epoch"], "best_epoch": state["best_epoch"],
    "train_minutes": round(state["train_seconds"] / 60, 1),
    "peak_gpu_mem_gb": round(torch.cuda.max_memory_allocated() / 1e9, 2) if DEVICE.type == "cuda" else None,
    "dev": dev_result, "sections": {k: v["per_label_crossfit"]["macro_f1"] for k, v in section_results.items()},
    "test_setting": list(TEST_SETTING), "test_statutes_per_case": float(sizes.mean()),
}
json.dump(summary, open(OUT_DIR / "run_summary.json", "w"), indent=2)
print(json.dumps(summary, indent=2))